In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.fft import fft

In [2]:
# Path to your merged sensor data
data_path = Path("../processed_data/merged_sensor_data.csv")

data = pd.read_csv(data_path)
print("Columns:", data.columns)
print("Recordings found:", data['recording'].unique())

Columns: Index(['time_x', 'seconds_elapsed', 'accel_z', 'accel_y', 'accel_x', 'time_y',
       'gyro_z', 'gyro_y', 'gyro_x', 'recording'],
      dtype='object')
Recordings found: ['walking_012' 'walking_009' 'walking_007' 'walking_001' 'walking_006'
 'walking_008' 'walking_011' 'walking_010' 'walking_003' 'walking_004'
 'walking_005' 'walking_002' 'jumping_009' 'jumping_007' 'jumping_001'
 'jumping_006' 'jumping_008' 'jumping_012' 'jumping_003' 'jumping_004'
 'jumping_005' 'jumping_002' 'jumping_011' 'jumping_010' 'standing_005'
 'standing_002' 'standing_003' 'standing_004' 'standing_010'
 'standing_011' 'standing_001' 'standing_006' 'standing_008'
 'standing_009' 'standing_007' 'standing_012' 'still_012' 'still_007'
 'still_009' 'still_008' 'still_001' 'still_006' 'still_011' 'still_010'
 'still_003' 'still_004' 'still_005' 'still_002'
 'jumping05-ange-2026-03-06_09-51-43' 'jump11-ange-2026-03-06_09-59-13'
 'standing03-ange-2026-03-06_09-19-11']


In [3]:
# Keep only relevant columns and rename to standard names
data = data.rename(columns={
    'accel_x': 'accel_x',
    'accel_y': 'accel_y',
    'accel_z': 'accel_z',
    'gyro_x': 'gyro_x',
    'gyro_y': 'gyro_y',
    'gyro_z': 'gyro_z'
})

# Keep only useful columns
data = data[['seconds_elapsed','accel_x','accel_y','accel_z',
             'gyro_x','gyro_y','gyro_z','recording']]

print(data.head())

   seconds_elapsed   accel_x   accel_y   accel_z    gyro_x    gyro_y  \
0         0.099765  1.704084  0.664405 -0.620752  0.274288 -0.395068   
1         0.109792  2.292617  0.927186 -1.084056  0.314669 -0.405201   
2         0.119818  2.741096  1.166143 -1.283366  0.392578 -0.312417   
3         0.129845  3.062587  1.361918 -1.554028  0.512527 -0.109067   
4         0.139871  2.967145  1.466173 -1.590161  0.684600  0.227399   

     gyro_z    recording  
0 -0.179186  walking_012  
1 -0.070252  walking_012  
2  0.066648  walking_012  
3  0.203660  walking_012  
4  0.310652  walking_012  


In [4]:
# Automatic mapping based on folder name
def map_activity(recording_name):
    name = recording_name.lower()
    if 'walk' in name:
        return 'walking'
    elif 'stand' in name:
        return 'standing'
    elif 'jump' in name:
        return 'jumping'
    elif 'still' in name:
        return 'still'
    else:
        return 'unknown'

data['activity'] = data['recording'].apply(map_activity)
print(data['activity'].value_counts())

activity
standing    25781
walking     24627
still       23819
jumping     23557
Name: count, dtype: int64


In [5]:
def create_windows(df, window_size, step_size):
    """
    Split dataframe into overlapping windows.
    """
    windows = []
    for start in range(0, len(df) - window_size + 1, step_size):
        window = df.iloc[start:start+window_size]
        windows.append(window)
    return windows

In [6]:
def extract_time_features(window):
    features = {}
    axes = ["accel_x","accel_y","accel_z","gyro_x","gyro_y","gyro_z"]
    
    for axis in axes:
        features[f"{axis}_mean"] = np.mean(window[axis])
        features[f"{axis}_std"] = np.std(window[axis])
        features[f"{axis}_var"] = np.var(window[axis])
    
    # Signal Magnitude Area (SMA) for accelerometer
    features["accel_sma"] = np.sum(
        np.abs(window["accel_x"]) + np.abs(window["accel_y"]) + np.abs(window["accel_z"])
    ) / len(window)
    
    # Correlations between accelerometer axes
    features["accel_xy_corr"] = np.corrcoef(window["accel_x"], window["accel_y"])[0,1]
    features["accel_xz_corr"] = np.corrcoef(window["accel_x"], window["accel_z"])[0,1]
    features["accel_yz_corr"] = np.corrcoef(window["accel_y"], window["accel_z"])[0,1]
    
    return features

In [7]:
def extract_frequency_features(window):
    features = {}
    for axis in ["accel_x","accel_y","accel_z"]:
        sig = window[axis].values
        fft_vals = np.abs(fft(sig))
        features[f"{axis}_dom_freq"] = np.argmax(fft_vals)
    return features

In [8]:
window_size = 50  # ~1 second
step_size = 25    # 50% overlap

all_features = []

for recording_id, group in data.groupby('recording'):

    # Skip short recordings
    if len(group) < window_size:
        print("Skipping short recording:", recording_id)
        continue

    windows = create_windows(group, window_size, step_size)

    for window in windows:
        time_feats = extract_time_features(window)
        freq_feats = extract_frequency_features(window)
        features = {**time_feats, **freq_feats}
        features['activity'] = window['activity'].iloc[0]
        features['recording'] = recording_id
        all_features.append(features)

print("Total windows processed:", len(all_features))

Total windows processed: 3839


In [9]:
features_df = pd.DataFrame(all_features)
print("Feature dataset shape:", features_df.shape)
features_df.head()

Feature dataset shape: (3839, 27)


,accel_x_mean,accel_x_std,accel_x_var,accel_y_mean,accel_y_std,accel_y_var,accel_z_mean,accel_z_std,accel_z_var,gyro_x_mean,...,gyro_z_var,accel_sma,accel_xy_corr,accel_xz_corr,accel_yz_corr,accel_x_dom_freq,accel_y_dom_freq,accel_z_dom_freq,activity,recording
0,-0.837983,1.758604,3.092689,0.171601,0.772426,0.596642,0.995800,2.133551,4.552041,-0.855934,...,0.444917,3.767699,0.378335,0.393158,0.446090,0,1,1,jumping,jump11-ange-2026-03-06_09-59-13
1,0.014290,2.361475,5.576565,-0.503273,1.993134,3.972584,-2.278114,5.302432,28.115784,-0.392847,...,0.349308,7.963501,-0.174241,-0.030494,0.498210,2,1,1,jumping,jump11-ange-2026-03-06_09-59-13
2,0.991794,1.638522,2.684755,-0.887473,1.829146,3.345774,-0.239770,7.099894,50.408498,-0.123881,...,0.155363,9.317532,-0.063957,0.238329,0.440889,0,0,1,jumping,jump11-ange-2026-03-06_09-59-13
3,0.655566,1.215642,1.477785,-0.893239,1.913605,3.661884,0.307948,6.883720,47.385596,0.306612,...,0.380792,8.921468,-0.231635,0.285444,0.535261,1,0,1,jumping,jump11-ange-2026-03-06_09-59-13
4,-0.018417,1.012740,1.025643,-0.530605,2.308746,5.330307,0.093246,6.671704,44.511641,0.076820,...,0.144026,8.833271,-0.344359,-0.027778,0.675745,2,1,1,jumping,jump11-ange-2026-03-06_09-59-13


In [10]:
features_df.to_csv("../processed_data/training_features.csv", index=False)
print("Feature dataset saved successfully!")

Feature dataset saved successfully!
